# Mistral AI y Cohere — Notebook interactivo

Modelos europeos con fuerte soporte multilingüe: Mistral para generación y Cohere para
embeddings, reranking y clasificación empresarial.

In [ ]:
# pip install mistralai cohere python-dotenv
import os
from mistralai import Mistral
import cohere

mistral = Mistral(api_key=os.getenv('MISTRAL_API_KEY', ''))
co = cohere.ClientV2(api_key=os.getenv('COHERE_API_KEY', ''))

MODELO_MISTRAL = 'mistral-small-latest'
MODELO_MISTRAL_MED = 'mistral-medium-latest'
print('Clientes Mistral y Cohere listos')

## 1. Mistral — Chat completions

In [ ]:
def chat_mistral(mensaje: str, system: str = '', modelo: str = MODELO_MISTRAL) -> str:
    mensajes = []
    if system:
        mensajes.append({'role': 'system', 'content': system})
    mensajes.append({'role': 'user', 'content': mensaje})

    resp = mistral.chat.complete(
        model=modelo,
        messages=mensajes,
        max_tokens=512,
        temperature=0.3,
    )
    return resp.choices[0].message.content

try:
    respuesta = chat_mistral(
        'Explica qué es Mistral AI y en qué se diferencia de otros proveedores.',
        system='Responde en español de forma concisa.'
    )
    print(respuesta)
except Exception as e:
    print(f'Error (configura MISTRAL_API_KEY): {e}')
    print('Obtén tu clave en: https://console.mistral.ai/')

## 2. Mistral — Ventajas multilingüe

In [ ]:
TEXTO_LEGAL_ES = """
El arrendatario se compromete a abonar la renta mensual dentro de los cinco primeros
días de cada mes. En caso de impago, el arrendador podrá iniciar el procedimiento
de desahucio conforme a la Ley de Arrendamientos Urbanos (LAU). La fianza depositada
equivale a dos mensualidades y será devuelta en el plazo de 30 días tras la finalización
del contrato, previa inspección del inmueble.
"""

def analizar_texto_legal(texto: str) -> dict:
    import json
    prompt = f"""Analiza este fragmento de contrato legal y responde en JSON:
{{
  "tipo_documento": "...",
  "partes": ["lista de partes mencionadas"],
  "obligaciones": ["lista de obligaciones"],
  "plazos": {{"nombre": "valor"}},
  "riesgos": ["posibles riesgos legales"]
}}

TEXTO:
{texto}"""
    try:
        resp = chat_mistral(prompt, modelo='mistral-medium-latest')
        # Limpiar markdown si lo envuelve
        if '```' in resp:
            resp = resp.split('```')[1].replace('json', '').strip()
        return json.loads(resp)
    except Exception as e:
        return {'error': str(e), 'raw': resp if 'resp' in dir() else ''}

try:
    analisis = analizar_texto_legal(TEXTO_LEGAL_ES)
    import json
    print(json.dumps(analisis, indent=2, ensure_ascii=False))
except Exception as e:
    print(f'Error: {e}')

## 3. Mistral — Function calling

In [ ]:
import json

HERRAMIENTAS_MISTRAL = [
    {
        'type': 'function',
        'function': {
            'name': 'consultar_expediente',
            'description': 'Consulta el estado de un expediente legal por su número.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'numero_expediente': {'type': 'string', 'description': 'Número de expediente, ej: EXP-2026-001'},
                },
                'required': ['numero_expediente'],
            },
        },
    }
]

EXPEDIENTES_MOCK = {
    'EXP-2026-001': {'estado': 'En tramitación', 'fase': 'Instrucción', 'fecha_resolucion': '2026-09-15'},
    'EXP-2026-042': {'estado': 'Resuelto', 'fase': 'Finalizado', 'fecha_resolucion': '2026-03-01'},
}

try:
    resp = mistral.chat.complete(
        model=MODELO_MISTRAL,
        messages=[{'role': 'user', 'content': '¿Cuál es el estado del expediente EXP-2026-001?'}],
        tools=HERRAMIENTAS_MISTRAL,
        tool_choice='auto',
    )

    if resp.choices[0].finish_reason == 'tool_calls':
        tc = resp.choices[0].message.tool_calls[0]
        args = json.loads(tc.function.arguments)
        print(f'Mistral solicita: {tc.function.name}({args})')

        resultado = EXPEDIENTES_MOCK.get(args['numero_expediente'], {'error': 'No encontrado'})

        resp_final = mistral.chat.complete(
            model=MODELO_MISTRAL,
            messages=[
                {'role': 'user', 'content': '¿Cuál es el estado del expediente EXP-2026-001?'},
                resp.choices[0].message,
                {'role': 'tool', 'tool_call_id': tc.id, 'content': json.dumps(resultado)},
            ],
        )
        print('Respuesta:', resp_final.choices[0].message.content)
except Exception as e:
    print(f'Error: {e}')

## 4. Cohere — Embeddings multilingüe

In [ ]:
import numpy as np

def obtener_embeddings_cohere(textos: list[str], tipo: str = 'search_document') -> list[np.ndarray]:
    resp = co.embed(
        texts=textos,
        model='embed-multilingual-v3.0',
        input_type=tipo,
        embedding_types=['float'],
    )
    return [np.array(emb) for emb in resp.embeddings.float_]

def similitud_coseno(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Búsqueda semántica multilingüe
documentos = [
    'El contrato de arrendamiento establece las condiciones del alquiler.',
    'La cláusula de rescisión permite terminar el contrato anticipadamente.',
    'Los honorarios del abogado se calcularán según el arancel vigente.',
    'The lease agreement sets the rental conditions.',   # inglés
    'Le contrat de bail définit les conditions de location.',  # francés
]

try:
    embs_docs = obtener_embeddings_cohere(documentos, 'search_document')

    query = 'rental contract termination'  # búsqueda en inglés sobre documentos en español
    emb_query = obtener_embeddings_cohere([query], 'search_query')[0]

    scores = [(similitud_coseno(emb_query, emb), doc) for emb, doc in zip(embs_docs, documentos)]
    scores.sort(reverse=True)

    print(f'Query en inglés: "{query}"')
    print('Resultados (multilingüe):')
    for score, doc in scores:
        print(f'  {score:.4f} — {doc}')
except Exception as e:
    print(f'Error (configura COHERE_API_KEY): {e}')

## 5. Cohere — Reranking

In [ ]:
def reranking_cohere(query: str, documentos: list[str], top_n: int = 3) -> list[dict]:
    resp = co.rerank(
        model='rerank-multilingual-v3.0',
        query=query,
        documents=documentos,
        top_n=top_n,
    )
    return [
        {
            'posicion': i + 1,
            'score': round(r.relevance_score, 4),
            'documento': documentos[r.index],
        }
        for i, r in enumerate(resp.results)
    ]

# Documentos de un sistema de soporte técnico
docs_soporte = [
    'Para restablecer la contraseña, ve a Configuración > Seguridad > Cambiar contraseña.',
    'El sistema de facturación se actualiza cada primer día de mes.',
    'Si olvidaste tu contraseña, haz clic en "¿Olvidaste tu contraseña?" en la pantalla de inicio de sesión.',
    'Los usuarios premium tienen acceso a soporte 24/7 por chat.',
    'Para recuperar el acceso a tu cuenta, también puedes contactar con soporte@empresa.com.',
    'El plan básico no incluye copias de seguridad automáticas.',
]

try:
    resultados = reranking_cohere('¿Cómo recupero mi contraseña?', docs_soporte)
    print('Top 3 documentos relevantes:')
    for r in resultados:
        print(f'  #{r["posicion"]} (score: {r["score"]}) — {r["documento"]}')
except Exception as e:
    print(f'Error: {e}')

## 6. Cohere — Clasificación de texto

In [ ]:
# Clasificación few-shot con Cohere (sin fine-tuning)
EJEMPLOS_CLASIFICACION = [
    cohere.ClassifyExample(text='Necesito cancelar mi suscripción', label='cancelacion'),
    cohere.ClassifyExample(text='¿Cuándo renueva mi plan?', label='facturacion'),
    cohere.ClassifyExample(text='La aplicación no carga', label='soporte_tecnico'),
    cohere.ClassifyExample(text='Quiero darme de baja', label='cancelacion'),
    cohere.ClassifyExample(text='Me han cobrado dos veces', label='facturacion'),
    cohere.ClassifyExample(text='El botón no funciona', label='soporte_tecnico'),
    cohere.ClassifyExample(text='¿Cómo cambio mi plan?', label='facturacion'),
    cohere.ClassifyExample(text='Error 500 al iniciar sesión', label='soporte_tecnico'),
]

TEXTOS_A_CLASIFICAR = [
    'Ya no me interesa el servicio',
    'No puedo descargar el archivo PDF',
    '¿El descuento se aplica en la próxima factura?',
]

try:
    resp = co.classify(
        model='embed-multilingual-v3.0',
        inputs=TEXTOS_A_CLASIFICAR,
        examples=EJEMPLOS_CLASIFICACION,
    )
    print('Clasificaciones:')
    for texto, clasificacion in zip(TEXTOS_A_CLASIFICAR, resp.classifications):
        print(f'  "{texto}"')
        print(f'    → {clasificacion.prediction} (confianza: {clasificacion.confidence:.2%})')
except Exception as e:
    print(f'Error: {e}')